# CISM Tutorial 02: FANMOD+, Initialization, And Stringency Selection

This notebook starts from the output of tutorial 01.

By the end of this notebook you should have:

- a validated prepared dataset folder
- a loaded `CISM` object
- a `TissueStateDiscriminativeMotifs` object
- a discriminative-motifs-versus-stringency plot
- an AUC-based choice of stringency parameters
- a direct asymmetric majority/minority stringency analysis
- a serialized artifact you can load in later analysis notebooks

## What This Notebook Covers

This notebook performs the transition from prepared graph txt files to an analysis-ready CISM state.

It will:

1. reuse the `network_dataset_root_path` and `dataset_folder` from tutorial 01
2. resolve the FANMOD+ binary already bundled in this repository
3. create tutorial-local `fanmod_output` and `fanmod_cache` folders
4. initialize `CISM`
5. load the prepared dataset
6. construct a discriminator
7. inspect discriminative motifs versus stringency
8. choose a stringency parameter using Random Forest ROC AUC
9. save a reusable pickle artifact for downstream notebooks

## Notes About FANMOD+ Binaries

This tutorial uses the FANMOD+ binaries already stored under `cism/FANMOD_binaries/`.

For tutorial purposes this is simpler and safer than moving them now, because the notebook can stay fully self-contained and directly point `CISM` to the repo-bundled executable.

In [ ]:
from pathlib import Path
import pickle
import platform

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

from cism import (
    CISM,
    DiscriminativeFeatureKey,
    HardDiscriminativeFC,
    TissueStateDiscriminativeMotifs,
    build_patient_motif_matrix,
    validate_network_dataset_directory,
)


In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "tutorials" else Path.cwd().resolve()
TUTORIALS_DIR = PROJECT_ROOT / "melanoma_tutorials"
TUTORIAL_RUNTIME_DIR = TUTORIALS_DIR / "runtime"
FANMOD_OUTPUT_ROOT = TUTORIAL_RUNTIME_DIR / "fanmod_output"
FANMOD_CACHE_ROOT = TUTORIAL_RUNTIME_DIR / "fanmod_cache"
SERIALIZED_ROOT = TUTORIAL_RUNTIME_DIR / "serialized"
ASYMMETRIC_STRINGENCY_ROOT = TUTORIAL_RUNTIME_DIR / "asymmetric_stringency"

FANMOD_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FANMOD_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
SERIALIZED_ROOT.mkdir(parents=True, exist_ok=True)
ASYMMETRIC_STRINGENCY_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Tutorial runtime directory: {TUTORIAL_RUNTIME_DIR}")
print(f"fanmod_output_root_path: {FANMOD_OUTPUT_ROOT}")
print(f"fanmod_cache_root_path: {FANMOD_CACHE_ROOT}")
print(f"serialized artifact directory: {SERIALIZED_ROOT}")


## Paste The Handoff Values From Tutorial 01

At the end of tutorial 01, the notebook printed these exact values:

- `network_dataset_root_path`
- `dataset_folder`

Use them directly here.

In [ ]:
# Replace these with the values printed at the end of tutorial 01.
network_dataset_root_path = str(PROJECT_ROOT / "pet_data")
dataset_folder = "Melanoma"  # Change this to the name of your dataset folder if it's different.

dataset_dir = Path(network_dataset_root_path) / dataset_folder
print(f"network_dataset_root_path = {network_dataset_root_path}")
print(f"dataset_folder = {dataset_folder}")
print(f"dataset_dir = {dataset_dir}")


In [ ]:
# Confirm that tutorial 01 produced a valid folder.
txt_files = validate_network_dataset_directory(dataset_dir, require_patient_class=True)
print(f"Validated {len(txt_files)} txt graph files in {dataset_dir}")


## Resolve The FANMOD+ Binary

The `CISM` object expects:

- `fanmod_path`: directory containing the binary
- `fanmod_exe`: the executable filename

In [ ]:
fanmod_path = PROJECT_ROOT/ ".." / "FANMOD_binaries"
if platform.system().lower().startswith("win"):
    fanmod_exe = "LocalFANMOD - Windows"
else:
    fanmod_exe = "LocalFANMOD"

fanmod_binary_path = fanmod_path / fanmod_exe
print(f"fanmod_path = {fanmod_path}")
print(f"fanmod_exe = {fanmod_exe}")
print(f"fanmod binary exists: {fanmod_binary_path.exists()}")
!chmod +x {fanmod_binary_path}  # Ensure the binary is executable.


## Initialize `CISM`

This tutorial uses the requested defaults:

- `motif_size = 4`
- `iterations = 1000`

In [ ]:
cism = CISM(
    fanmod_exe=fanmod_exe,
    fanmod_path=str(fanmod_path),
    network_dataset_root_path=network_dataset_root_path,
    fanmod_output_root_path=str(FANMOD_OUTPUT_ROOT),
    fanmod_cache_root_path=str(FANMOD_CACHE_ROOT),
    motif_size=4,
    iterations=1000,
)

cism


## Load The Prepared Dataset

Use the requested defaults when adding the dataset:

- `force_run_fanmod=False`
- `force_parse=False`
- `n_jobs=1` to avoid memory-heavy process pools on the large melanoma motif table

In [ ]:
# Replace dataset_type and dataset_name with the metadata you want in downstream analysis.
dataset_type = "Disease"
dataset_name = "Melanoma"

# Set these True after changing graph preparation or cell-type mapping so stale FANMOD/cache
# artifacts cannot be reused. After one clean rebuild, you may switch them back to False.
force_run_fanmod = False
force_parse = False
DATASET_N_JOBS = 1

cism.add_dataset(
    dataset_folder=dataset_folder,
    dataset_type=dataset_type,
    dataset_name=dataset_name,
    force_run_fanmod=force_run_fanmod,
    force_parse=force_parse,
    n_jobs=DATASET_N_JOBS,
    require_patient_class=True,
)


In [ ]:
motif_df = cism.motif_dataset()
print(f"Loaded motif rows: {0 if motif_df is None else len(motif_df)}")
display(motif_df.head() if motif_df is not None else None)


## Define The Analysis Metadata Needed For Discrimination

To analyze discriminative motifs, we need:

- a path to `patient_class.csv`
- a mapping from raw class values to readable labels
- a mapping from integer cell-type ids to cell-type names

Both mappings are dataset-specific, so replace the examples below with your real values.

In [ ]:
# This melanoma analysis focuses on the NN versus NP patient classes.
# Keep the survival helper below only for datasets that need a numeric survival split.
survival_rate_cutoff = 1000


def map_survival_rate_to_class(survival_rate: int):
    if survival_rate < survival_rate_cutoff:
        return "Short-term survival"
    return "Long-term survival"


labels = ["NN", "NP"]


In [ ]:
tissue_state_csv_path = str(dataset_dir / "patient_class.csv")

# Use this mapping when patient_class.csv already contains readable categorical labels.
tissue_state_to_string = {
    "NP": "NP",
    "NN": "NN",
    "PN": "PN",
    "PP": "PP"
}

# Paste from tutorial 01 output. This is the graph color id -> cell-type name mapping.
common_cells_type = {0: 'DC', 1: 'b cell', 2: 'blood vessels', 3: 'cd4 apc', 4: 'cd4 t cell', 5: 'cd8 t cell', 6: 'gc', 7: 'hevs', 8: 'mac', 9: 'memory%cd4 t cell', 10: 'neutrophil', 11: 'nk cell', 12: 'stroma', 13: 'treg', 14: 'tumor', 15: 'unidentified'}

patient_class_df = pd.read_csv(tissue_state_csv_path, index_col=0, names=["class"])
analysis_patient_class_df = patient_class_df[patient_class_df["class"].isin(labels)].copy()
motif_patient_uids = set(motif_df["Patient_uId"].astype(str).unique())
missing_tissue_state = sorted(motif_patient_uids - set(patient_class_df.index.astype(str)))

print(f"Patients with motif rows: {len(motif_patient_uids)}")
print(f"Patients with tissue-state labels: {len(patient_class_df)}")
print(f"Patients in analysis labels {labels}: {len(analysis_patient_class_df)}")
display(analysis_patient_class_df["class"].value_counts().rename("patient_count"))
print(f"Patients without tissue-state labels and excluded from discrimination: {missing_tissue_state}")
print(f"labels = {labels}")
print("common_cells_type =")
print(common_cells_type)


In [ ]:
discriminator = TissueStateDiscriminativeMotifs(
    cism=cism,
    tissue_state_csv_path=tissue_state_csv_path,
    tissue_state_to_string= tissue_state_to_string,  # Use tissue_state_to_string when labels are already categorical.
    common_cells_type=common_cells_type,
    #tissue_state_func=map_survival_rate_to_class,
    
)

display(discriminator.get_patients_class(classes=labels).groupby("patient_class").count())


## Plot Discriminative Motifs Per Stringency

This gives a first view of how many motifs remain as the discrimination stringency parameter increases.

Replace the colors if you prefer a different class palette.

In [ ]:
#warnings.filterwarnings("ignore", category=FutureWarning, module="seaborn")

class_to_color = {
    "NN": "tab:blue",
    "NP": "tab:orange",
    "PN": "tab:green",
    "PP": "tab:red",
}

discover_result = discriminator.discover(
    extract_by=DiscriminativeFeatureKey.STRUCTURE_AND_CELL_IDENTITIES,
    classes=labels,
)

discover_result.plot_number_of_motifs_versus_discrimination_stringency_parameter(
    class_to_color=class_to_color,
)


## Define A Feature Configuration Template

We will tune the main stringency controls used by the Random Forest validation:

- `shared_percentage`
- `max_class_features`

The configuration below is only the starting template. Optuna will search around it.

In [ ]:
feature_conf_template = HardDiscriminativeFC(
    labels=labels,
    extract_by=DiscriminativeFeatureKey.STRUCTURE_AND_CELL_IDENTITIES,
    use_cells_type_composition=False,
    use_motifs=True,
    shared_percentage=0.5,
    max_class_features=10,
)

feature_conf_template


## Choose The Best Stringency By Random Forest ROC AUC

This step uses the built-in Optuna tuner to search for a good stringency setting.

The objective is `roc_auc`, which is computed from the existing CISM motif-analysis pipeline.

In [ ]:
# Avoid process-based parallelism here: the serialized CISM/motif state is large,
# and loky workers can be killed by memory pressure when n_jobs is high.
STRINGENCY_N_JOBS = 1
STRINGENCY_PREFER = "threads"

tuning_result = discriminator.tune_stringency(
    feature_conf=feature_conf_template,
    n_trials=3,
    metric="roc_auc",
    random_state=0,
    n_jobs=STRINGENCY_N_JOBS,
    prefer=STRINGENCY_PREFER,
    shared_percentage_range=(0.35, 0.5),
    max_class_features_range=(30, 30),
)

print("Best parameters:")
print(tuning_result.best_params)
print(f"Best ROC AUC: {tuning_result.best_score:.4f}")


In [ ]:
tuning_trials_df = tuning_result.trials_dataframe()
display(tuning_trials_df)


## Build The Selected Analysis Configuration

We now convert the tuned parameters into the feature configuration we will carry into later notebooks.

In [ ]:
best_feature_conf = HardDiscriminativeFC(
    labels=labels,
    extract_by=DiscriminativeFeatureKey.STRUCTURE_AND_CELL_IDENTITIES,
    use_cells_type_composition=False,
    use_motifs=True,
    shared_percentage=tuning_result.best_params["shared_percentage"],
    max_class_features=tuning_result.best_params["max_class_features"],
)

best_feature_conf


## Asymmetric Discriminative Motif Stringency

The standard hard discriminative strategy is very strict about the opposite class. Here we keep the majority-class recurrence requirement, but allow a motif to appear in the minority class up to a tunable maximum.

This is independent of the soft motif scoring in tutorial 03b. It queries the CISM motif table directly, builds a patient-by-motif frequency matrix, and selects motifs by patient-level prevalence:

- `majority_min`: minimum prevalence in the motif's dominant class.
- `minority_max`: maximum allowed prevalence in the other class.

The direct query cell below answers: “which motifs fit these criteria?” The tuning cells then choose thresholds by leakage-safe LOOCV Random Forest ROC AUC.

In [ ]:
def prepare_asymmetric_motif_matrix(discriminator, labels, p_value_threshold=0.05):
    patient_classes = discriminator.get_patients_class(labels)
    patient_classes = patient_classes.loc[~patient_classes.index.duplicated(keep="first")]
    motif_table = discriminator.cism.motifs_dataset.copy()
    if p_value_threshold is not None:
        motif_table = motif_table[motif_table["p_value"] < p_value_threshold]
    matrix = build_patient_motif_matrix(
        motif_table,
        patient_ids=patient_classes.index,
        motif_col="ID",
    )
    labels_series = patient_classes["patient_class"].reindex(matrix.index)
    return matrix, labels_series


def compute_asymmetric_prevalence(matrix, labels_series, label_order):
    label_0, label_1 = label_order
    group_0 = matrix.loc[labels_series == label_0]
    group_1 = matrix.loc[labels_series == label_1]

    prevalence_0 = (group_0 > 0).mean(axis=0).fillna(0.0)
    prevalence_1 = (group_1 > 0).mean(axis=0).fillna(0.0)
    mean_0 = group_0.mean(axis=0).fillna(0.0)
    mean_1 = group_1.mean(axis=0).fillna(0.0)
    group_1_dominant = (prevalence_1 > prevalence_0) | (
        np.isclose(prevalence_1, prevalence_0) & (mean_1 > mean_0)
    )

    prevalence = pd.DataFrame({"ID": matrix.columns})
    prevalence[f"{label_0}_prevalence"] = prevalence_0.to_numpy(dtype=float)
    prevalence[f"{label_1}_prevalence"] = prevalence_1.to_numpy(dtype=float)
    prevalence[f"{label_0}_mean_frequency"] = mean_0.to_numpy(dtype=float)
    prevalence[f"{label_1}_mean_frequency"] = mean_1.to_numpy(dtype=float)
    prevalence["dominant_group"] = np.where(group_1_dominant, label_1, label_0)
    prevalence["minority_group"] = np.where(group_1_dominant, label_0, label_1)
    prevalence["dominant_prevalence"] = np.where(group_1_dominant, prevalence_1, prevalence_0)
    prevalence["minority_prevalence"] = np.where(group_1_dominant, prevalence_0, prevalence_1)
    prevalence["dominant_mean_frequency"] = np.where(group_1_dominant, mean_1, mean_0)
    prevalence["minority_mean_frequency"] = np.where(group_1_dominant, mean_0, mean_1)
    prevalence["prevalence_gap"] = prevalence["dominant_prevalence"] - prevalence["minority_prevalence"]
    prevalence["frequency_gap"] = prevalence["dominant_mean_frequency"] - prevalence["minority_mean_frequency"]
    return prevalence


def select_asymmetric_motifs(matrix, labels_series, label_order, majority_min, minority_max, top_k):
    if not 0.0 <= minority_max < majority_min <= 1.0:
        raise ValueError("Expected 0 <= minority_max < majority_min <= 1.")
    prevalence = compute_asymmetric_prevalence(matrix, labels_series, label_order)
    selected = prevalence[
        (prevalence["dominant_prevalence"] >= majority_min)
        & (prevalence["minority_prevalence"] <= minority_max)
        & (prevalence["prevalence_gap"] > 0)
    ].copy()
    selected = selected.sort_values(
        ["prevalence_gap", "dominant_prevalence", "frequency_gap"],
        ascending=False,
    ).head(top_k)
    return selected["ID"].tolist(), selected


def asymmetric_metrics(results, positive_label):
    valid = results[results["valid_fold"]].copy()
    if valid.empty:
        return {
            "roc_auc": np.nan,
            "accuracy": np.nan,
            "balanced_accuracy": np.nan,
            "valid_folds": 0,
            "empty_folds": int((~results["valid_fold"]).sum()),
            "mean_selected_motifs": float(results["cFeatures"].mean()) if not results.empty else 0.0,
        }
    y_true = valid["class"].tolist()
    y_pred = valid["pred_class"].tolist()
    y_binary = [1 if label == positive_label else 0 for label in y_true]
    probabilities = valid["positive_probability"].to_numpy(dtype=float)
    return {
        "roc_auc": float(roc_auc_score(y_binary, probabilities)) if len(set(y_binary)) == 2 else np.nan,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)) if len(set(y_true)) == 2 else np.nan,
        "valid_folds": int(valid.shape[0]),
        "empty_folds": int((~results["valid_fold"]).sum()),
        "mean_selected_motifs": float(results["cFeatures"].mean()) if not results.empty else 0.0,
    }


def evaluate_asymmetric_stringency(matrix, labels_series, label_order, majority_min, minority_max, top_k, random_state=0):
    positive_label, negative_label = label_order
    rows = []
    selected_by_patient = {}
    for held_out_patient in matrix.index:
        train_index = matrix.index.difference([held_out_patient], sort=False)
        x_train_all = matrix.loc[train_index]
        y_train = labels_series.loc[train_index]
        x_test_all = matrix.loc[[held_out_patient]]
        y_test = labels_series.loc[held_out_patient]

        selected, _ = select_asymmetric_motifs(
            x_train_all,
            y_train,
            label_order,
            majority_min=majority_min,
            minority_max=minority_max,
            top_k=top_k,
        )
        selected_by_patient[str(held_out_patient)] = selected
        if not selected:
            rows.append({
                "patient_id": held_out_patient,
                "class": y_test,
                "pred_class": None,
                "positive_probability": np.nan,
                "cFeatures": 0,
                "valid_fold": False,
            })
            continue

        clf = RandomForestClassifier(random_state=random_state, n_jobs=-1)
        clf.fit(x_train_all.reindex(columns=selected, fill_value=0.0), y_train)
        pred = clf.predict(x_test_all.reindex(columns=selected, fill_value=0.0))[0]
        prob = clf.predict_proba(x_test_all.reindex(columns=selected, fill_value=0.0))[0]
        positive_index = int(np.where(clf.classes_ == positive_label)[0][0])
        rows.append({
            "patient_id": held_out_patient,
            "class": y_test,
            "pred_class": pred,
            "positive_probability": float(prob[positive_index]),
            "cFeatures": len(selected),
            "valid_fold": True,
        })

    results = pd.DataFrame(rows).set_index("patient_id")
    metrics = asymmetric_metrics(results, positive_label)
    metrics["majority_min"] = float(majority_min)
    metrics["minority_max"] = float(minority_max)
    return results, metrics, selected_by_patient


In [ ]:
asymmetric_matrix, asymmetric_labels = prepare_asymmetric_motif_matrix(
    discriminator,
    labels,
    p_value_threshold=0.05,
)

DIRECT_MAJORITY_MIN = best_feature_conf.shared_percentage
DIRECT_MINORITY_MAX = DIRECT_MAJORITY_MIN / 2
ASYMMETRIC_TOP_K = best_feature_conf.max_class_features

asymmetric_selected_ids, asymmetric_selected_table = select_asymmetric_motifs(
    asymmetric_matrix,
    asymmetric_labels,
    labels,
    majority_min=DIRECT_MAJORITY_MIN,
    minority_max=DIRECT_MINORITY_MAX,
    top_k=ASYMMETRIC_TOP_K,
)

print(f"Patient-by-motif matrix: {asymmetric_matrix.shape[0]} patients x {asymmetric_matrix.shape[1]:,} motifs")
print(f"Direct asymmetric query: majority_min={DIRECT_MAJORITY_MIN:.4f}, minority_max={DIRECT_MINORITY_MAX:.4f}")
print(f"Selected motifs: {len(asymmetric_selected_ids)}")
display(asymmetric_selected_table.head(20))


In [ ]:
def tune_asymmetric_stringency(
    matrix,
    labels_series,
    label_order,
    top_k,
    n_trials,
    random_state=0,
    majority_range=(0.35, 0.95),
    fixed_majority_min=None,
):
    import optuna

    majority_low, majority_high = majority_range
    sampler = optuna.samplers.TPESampler(seed=random_state)

    def objective(trial):
        if fixed_majority_min is None:
            majority_min = trial.suggest_float("majority_min", majority_low, majority_high)
            minority_upper = np.nextafter(majority_min, 0.0)
        else:
            majority_min = float(fixed_majority_min)
            trial.set_user_attr("majority_min", majority_min)
            minority_upper = majority_min / 2
        minority_max = trial.suggest_float("minority_max", 0.0, float(minority_upper))
        _, metrics, _ = evaluate_asymmetric_stringency(
            matrix,
            labels_series,
            label_order,
            majority_min=majority_min,
            minority_max=minority_max,
            top_k=top_k,
            random_state=random_state,
        )
        for key, value in metrics.items():
            trial.set_user_attr(key, value)
        return 0.0 if pd.isna(metrics["roc_auc"]) else metrics["roc_auc"]

    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials)
    trials_df = study.trials_dataframe()
    best_majority_min = float(fixed_majority_min if fixed_majority_min is not None else study.best_params["majority_min"])
    best_minority_max = float(study.best_params["minority_max"])
    final_results, final_metrics, selected_by_patient = evaluate_asymmetric_stringency(
        matrix,
        labels_series,
        label_order,
        majority_min=best_majority_min,
        minority_max=best_minority_max,
        top_k=top_k,
        random_state=random_state,
    )
    return study, trials_df, final_results, final_metrics, selected_by_patient


ASYMMETRIC_N_TRIALS = 20
ASYMMETRIC_MAJORITY_RANGE = (0.35, 0.95)

joint_asym_study, joint_asym_trials_df, joint_asym_results, joint_asym_metrics, joint_selected_by_patient = tune_asymmetric_stringency(
    asymmetric_matrix,
    asymmetric_labels,
    labels,
    top_k=ASYMMETRIC_TOP_K,
    n_trials=ASYMMETRIC_N_TRIALS,
    random_state=0,
    majority_range=ASYMMETRIC_MAJORITY_RANGE,
    fixed_majority_min=None,
)

display(pd.Series(joint_asym_metrics, name="joint_asymmetric_stringency"))
display(joint_asym_trials_df.sort_values("value", ascending=False).head(10))


In [ ]:
fixed_majority_min = best_feature_conf.shared_percentage
fixed_asym_study, fixed_asym_trials_df, fixed_asym_results, fixed_asym_metrics, fixed_selected_by_patient = tune_asymmetric_stringency(
    asymmetric_matrix,
    asymmetric_labels,
    labels,
    top_k=ASYMMETRIC_TOP_K,
    n_trials=ASYMMETRIC_N_TRIALS,
    random_state=0,
    majority_range=ASYMMETRIC_MAJORITY_RANGE,
    fixed_majority_min=fixed_majority_min,
)

print(f"Fixed majority_min={fixed_majority_min:.4f}; tuned minority_max in [0, {fixed_majority_min / 2:.4f}]")
display(pd.Series(fixed_asym_metrics, name="fixed_majority_asymmetric_stringency"))
display(fixed_asym_trials_df.sort_values("value", ascending=False).head(10))


In [ ]:
joint_best_ids, joint_best_table = select_asymmetric_motifs(
    asymmetric_matrix,
    asymmetric_labels,
    labels,
    majority_min=joint_asym_metrics["majority_min"],
    minority_max=joint_asym_metrics["minority_max"],
    top_k=ASYMMETRIC_TOP_K,
)
fixed_best_ids, fixed_best_table = select_asymmetric_motifs(
    asymmetric_matrix,
    asymmetric_labels,
    labels,
    majority_min=fixed_asym_metrics["majority_min"],
    minority_max=fixed_asym_metrics["minority_max"],
    top_k=ASYMMETRIC_TOP_K,
)

asymmetric_selected_table.to_csv(ASYMMETRIC_STRINGENCY_ROOT / "direct_asymmetric_selected_motifs.csv", index=False)
joint_asym_trials_df.to_csv(ASYMMETRIC_STRINGENCY_ROOT / "joint_asymmetric_stringency_trials.csv", index=False)
joint_asym_results.to_csv(ASYMMETRIC_STRINGENCY_ROOT / "joint_asymmetric_stringency_loocv_results.csv")
pd.Series(joint_asym_metrics, name="value").to_csv(ASYMMETRIC_STRINGENCY_ROOT / "joint_asymmetric_stringency_metrics.csv")
joint_best_table.to_csv(ASYMMETRIC_STRINGENCY_ROOT / "joint_asymmetric_selected_motifs.csv", index=False)
fixed_asym_trials_df.to_csv(ASYMMETRIC_STRINGENCY_ROOT / "fixed_majority_asymmetric_stringency_trials.csv", index=False)
fixed_asym_results.to_csv(ASYMMETRIC_STRINGENCY_ROOT / "fixed_majority_asymmetric_stringency_loocv_results.csv")
pd.Series(fixed_asym_metrics, name="value").to_csv(ASYMMETRIC_STRINGENCY_ROOT / "fixed_majority_asymmetric_stringency_metrics.csv")
fixed_best_table.to_csv(ASYMMETRIC_STRINGENCY_ROOT / "fixed_majority_asymmetric_selected_motifs.csv", index=False)

asymmetric_summary = {
    "direct_majority_min": DIRECT_MAJORITY_MIN,
    "direct_minority_max": DIRECT_MINORITY_MAX,
    "joint_best_majority_min": joint_asym_metrics["majority_min"],
    "joint_best_minority_max": joint_asym_metrics["minority_max"],
    "fixed_majority_min": fixed_asym_metrics["majority_min"],
    "fixed_best_minority_max": fixed_asym_metrics["minority_max"],
    "top_k": ASYMMETRIC_TOP_K,
    "n_trials": ASYMMETRIC_N_TRIALS,
}

print("Saved asymmetric stringency outputs to:")
print(ASYMMETRIC_STRINGENCY_ROOT)
asymmetric_summary


## Optional: Run One Final Analysis Pass With The Selected Stringency

This is not strictly required for moving forward, but it gives you a first concrete analysis object at the selected setting.

In [ ]:
best_analysis_result = discriminator.analyze_motifs(
    feature_conf=best_feature_conf,
    exclude_patients=[],
    n_jobs=STRINGENCY_N_JOBS,
    prefer=STRINGENCY_PREFER,
)

print(f"Selected-setting ROC AUC: {best_analysis_result.get_roc_auc_score():.4f}")
display(best_analysis_result.results.head())


## Save A Reusable Analysis-Ready Artifact

The main goal is to leave this notebook with something you can load directly in later analysis notebooks.

We save:

- the `CISM` object
- the discriminator
- the selected feature configuration
- the tuning summary table
- the key path/config metadata

In [ ]:
artifact_path = SERIALIZED_ROOT / f"{dataset_folder}_cism_ready.pkl"

artifact = {
    "cism": cism,
    "discriminator": discriminator,
    "best_feature_conf": best_feature_conf,
    "tuning_best_params": tuning_result.best_params,
    "tuning_best_score": tuning_result.best_score,
    "tuning_trials_df": tuning_trials_df,
    "asymmetric_stringency_summary": globals().get("asymmetric_summary"),
    "network_dataset_root_path": network_dataset_root_path,
    "dataset_folder": dataset_folder,
    "dataset_name": dataset_name,
    "fanmod_path": str(fanmod_path),
    "fanmod_exe": fanmod_exe,
    "fanmod_output_root_path": str(FANMOD_OUTPUT_ROOT),
    "fanmod_cache_root_path": str(FANMOD_CACHE_ROOT),
    "tissue_state_csv_path": tissue_state_csv_path,
    "tissue_state_to_string": tissue_state_to_string,
    "tissue_state_mode": "categorical_labels",
    "survival_rate_cutoff": survival_rate_cutoff,
    "common_cells_type": common_cells_type,
    "labels": labels,
    "missing_tissue_state": missing_tissue_state,
}

with open(artifact_path, "wb") as handle:
    pickle.dump(artifact, handle)

print(f"Saved analysis-ready artifact: {artifact_path}")


In [ ]:
print("Notebook 02 summary:")
print(f"network_dataset_root_path = {network_dataset_root_path}")
print(f"dataset_folder = {dataset_folder}")
print(f"fanmod_path = {fanmod_path}")
print(f"fanmod_exe = {fanmod_exe}")
print(f"fanmod_output_root_path = {FANMOD_OUTPUT_ROOT}")
print(f"fanmod_cache_root_path = {FANMOD_CACHE_ROOT}")
print(f"best stringency shared_percentage = {best_feature_conf.shared_percentage}")
print(f"best stringency max_class_features = {best_feature_conf.max_class_features}")
print(f"artifact_path = {artifact_path}")


## You Are Done When...

Before moving to the analysis notebook, confirm that:

- the dataset from tutorial 01 validated successfully
- FANMOD+ was resolved correctly from `FANMOD_binaries/`
- `CISM(...)` was initialized with `motif_size=4` and `iterations=1000`
- `cism.add_dataset(...)` completed after rebuilding stale FANMOD/cache artifacts
- the discriminative-motifs-versus-stringency plot was generated
- the best stringency was chosen from Random Forest ROC AUC
- the asymmetric majority/minority stringency analysis was saved
- the pickle artifact was saved successfully

At that point, you have something analysis-ready that later notebooks can load directly.
